# Chapter 05-01 · Predicting a number: the best constant

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** easy - fifteen numbers, all visible at once

**Prerequisites:** 04-02 for baselines, 03-01 for what the mean and median minimise.

**Position in the learning path:** module 05, chapter 1 of 12. The first chapter of the course that fits
a model, and the model has no features.

---

## Why this matters

Module 04 built the workflow. Module 05 starts predicting numbers, and it starts with the simplest
predictor there is: **one constant, the same answer for everybody.**

That sounds like a warm-up and it is not. It is the baseline every later model in this module will be
measured against - and, more interestingly, it is where a fact you have met twice becomes a decision you
have to make:

> **The best constant depends on which metric you are judged by.** Under squared error it is the mean.
> Under absolute error it is the median. Under other metrics it is neither.

Which means **choosing a metric is choosing a predictor**, before any modelling happens at all. 03-01
showed this as a property of summaries; 04-02 used it to build baselines. Here it becomes the thing the
chapter is about, and the two answers differ by enough to matter.

## What you will be able to do

- Find the best constant predictor for a metric, by search and by formula
- Explain why the median minimises absolute error and the mean minimises squared error, from the shape of
  the loss
- Predict what one outlier does to each, and check
- Choose a constant when the cost of being wrong is asymmetric
- Say what "the model must beat the baseline" means precisely, in the units of the target

## Warm-up: retrieve, do not reread

1. What is a skill score, and what does a negative one mean?
2. In 04-02, which baseline beat both models, and why?
3. Which constant minimises the total absolute error of a set of numbers?

<br>

*Answers: (1) `(baseline error - model error) / baseline error`; negative means worse than free. (2) each
member's own running mean - members differ a lot from each other and little from themselves. (3) the
median.*

## Fifteen deliveries

Small enough to print, small enough to check every calculation by hand. **These are minutes from order to
doorstep for fifteen deliveries.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# TINY, SYNTHETIC: 15 delivery times in minutes, sorted so you can see the shape
minutes = np.array([22, 25, 27, 28, 30, 31, 33, 34, 36, 38, 41, 44, 47, 52, 58])

print("the fifteen deliveries:", minutes)
print()
print("mean   %.4f minutes" % minutes.mean())
print("median %.1f minutes" % np.median(minutes))
print("they differ by %.2f minutes" % (minutes.mean() - np.median(minutes)))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(minutes, np.zeros_like(minutes), "o", color="#0072B2", markersize=11, alpha=0.75)
for value in minutes:
    ax.annotate(str(value), (value, 0), textcoords="offset points", xytext=(0, 13),
                ha="center", fontsize=8, color="#555555")
ax.axvline(np.median(minutes), color="#009E73", linewidth=2.5)
ax.text(np.median(minutes), -0.36, "median %.0f" % np.median(minutes), ha="center",
        color="#009E73", fontsize=10, fontweight="bold")
ax.axvline(minutes.mean(), color="#D55E00", linewidth=2.5)
ax.text(minutes.mean(), 0.30, "mean %.1f" % minutes.mean(), ha="center",
        color="#D55E00", fontsize=10, fontweight="bold")
ax.set_ylim(-0.55, 0.55)
ax.set_yticks([])
ax.set_xlabel("minutes from order to doorstep")
ax.set_title("Fifteen deliveries. The mean sits above the median, because the tail is on the right",
             fontsize=11)
plt.tight_layout()
plt.show()

The mean is **2.40 minutes higher** than the median, because the distribution has a right tail - a few
slow deliveries pull the average up and leave the middle where it is. That gap is the whole chapter.

## Which constant is best? Ask the metric

A **constant predictor** says the same number for every delivery. To find the best one, try them all and
keep the one with the smallest error - which is exactly 03-01's grid search, and here it is cheap enough
to draw.

**Predict before running:** where will each curve bottom out?

In [ ]:
candidates = np.linspace(15, 70, 2201)
absolute_error = np.array([np.abs(minutes - c).mean() for c in candidates])
squared_error = np.array([((minutes - c) ** 2).mean() for c in candidates])

best_absolute = candidates[absolute_error.argmin()]
best_squared = candidates[squared_error.argmin()]

print("searching 2,201 candidate constants between 15 and 70 minutes:")
print()
print("  best under mean ABSOLUTE error : %.3f   (the median is %.1f)"
      % (best_absolute, np.median(minutes)))
print("  best under mean SQUARED error  : %.3f   (the mean is %.4f)"
      % (best_squared, minutes.mean()))
print()
print("and the errors those constants achieve - the baseline every later model must beat:")
print("  mean absolute error : %.4f minutes" % absolute_error.min())
print("  mean squared error  : %.4f" % squared_error.min())

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))

left.plot(candidates, absolute_error, color="#009E73", linewidth=2.2)
left.axvline(np.median(minutes), color="#009E73", linestyle="--")
left.plot([best_absolute], [absolute_error.min()], "o", color="#000000", markersize=9)
left.plot(minutes, np.full_like(minutes, absolute_error.min() * 0.62, dtype=float), "|",
          color="#0072B2", markersize=13)
left.set_xlabel("candidate constant (minutes)")
left.set_ylabel("mean absolute error")
left.set_title("Absolute error bottoms out at the median, %.0f" % np.median(minutes), fontsize=11)

right.plot(candidates, squared_error, color="#D55E00", linewidth=2.2)
right.axvline(minutes.mean(), color="#D55E00", linestyle="--")
right.plot([best_squared], [squared_error.min()], "o", color="#000000", markersize=9)
right.plot(minutes, np.full_like(minutes, squared_error.min() * 0.62, dtype=float), "|",
           color="#0072B2", markersize=13)
right.set_xlabel("candidate constant (minutes)")
right.set_ylabel("mean squared error")
right.set_title("Squared error bottoms out at the mean, %.1f" % minutes.mean(), fontsize=11)

plt.tight_layout()
plt.show()

**Two metrics, two different best answers, on identical data.** The blue ticks along the bottom are the
fifteen deliveries, so you can see where each minimum sits relative to them.

The shapes are different too, and the difference is the reason.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
zoom = (candidates > 26) & (candidates < 44)
ax.plot(candidates[zoom], absolute_error[zoom], color="#009E73", linewidth=2.4,
        label="mean absolute error")
for value in minutes[(minutes > 26) & (minutes < 44)]:
    ax.axvline(value, color="#cccccc", linewidth=0.9, zorder=0)
ax.plot([np.median(minutes)], [absolute_error.min()], "o", color="#000000", markersize=9)
ax.text(np.median(minutes) + 0.35, absolute_error.min() + 0.07,
        "the minimum sits exactly on a data point", fontsize=9)
ax.set_xlabel("candidate constant (minutes)")
ax.set_ylabel("mean absolute error")
ax.set_title("Zoomed in: absolute error is straight lines with a kink at every delivery",
             fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Absolute error is made of straight segments, with a kink at each of the fifteen values** (the grey
lines). Between two deliveries the slope is constant, and it equals *how many points are above minus how
many are below*. So the curve stops falling exactly where those two counts balance - **which is the
definition of the median.**

**Squared error is a smooth parabola**, and a parabola's minimum is where the average residual is zero -
**which is the definition of the mean.**

That is the whole derivation, and it explains a property people find surprising: **the median-optimal
answer depends only on the *order* of the data, not on the values.** Move the largest delivery from 58 to
580 and the kink structure around the middle is unchanged. The mean has no such protection.

## One stuck driver

Delivery sixteen: a van breaks down and the order arrives after **195 minutes**. One value, out of
sixteen.

**Predict before running:** how far does each summary move?

In [ ]:
with_breakdown = np.append(minutes, 195)

comparison = pd.DataFrame([
    {"summary": "mean", "before": round(minutes.mean(), 2),
     "after": round(with_breakdown.mean(), 2),
     "moved by": round(with_breakdown.mean() - minutes.mean(), 2)},
    {"summary": "median", "before": float(np.median(minutes)),
     "after": float(np.median(with_breakdown)),
     "moved by": round(float(np.median(with_breakdown) - np.median(minutes)), 2)},
])
print(comparison.to_string(index=False))
print()
print("the mean moved %.1f times as far as the median"
      % ((with_breakdown.mean() - minutes.mean()) / (np.median(with_breakdown) - np.median(minutes))))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(with_breakdown[:-1], np.zeros(15), "o", color="#0072B2", markersize=10, alpha=0.75)
ax.plot([195], [0], "X", color="#D55E00", markersize=15)
ax.annotate("the breakdown", (195, 0), textcoords="offset points", xytext=(0, 18), ha="center",
            fontsize=9, color="#D55E00")

for label, before, after, colour, height in [
        ("median", np.median(minutes), np.median(with_breakdown), "#009E73", -0.30),
        ("mean", minutes.mean(), with_breakdown.mean(), "#D55E00", 0.30)]:
    ax.annotate("", xy=(after, height), xytext=(before, height),
                arrowprops=dict(arrowstyle="-|>", color=colour, linewidth=2.6))
    ax.plot([before], [height], "o", color=colour, markersize=7)
    ax.text((before + after) / 2, height + (0.13 if height > 0 else -0.19),
            "%s: %.1f to %.1f" % (label, before, after), ha="center", color=colour, fontsize=9.5,
            fontweight="bold")

ax.set_ylim(-0.62, 0.62)
ax.set_xlim(10, 215)
ax.set_yticks([])
ax.set_xlabel("minutes")
ax.set_title("One value in sixteen. The mean chases it; the median barely notices", fontsize=11.5)
plt.tight_layout()
plt.show()

**The mean moves 9.91 minutes. The median moves 1.0.** One value in sixteen dragged the squared-error
answer nearly ten minutes, and the absolute-error answer by a single minute.

That is not a defect of the mean - it is the mean doing its job. The breakdown sits 158.6 minutes from
the old mean and the fastest delivery sits 14.4 minutes from it, a ratio of 11. **Squaring turns that into
a ratio of 121**, so under squared error that single delivery carries as much weight as 121 ordinary ones,
and the best constant moves towards it. If a three-hour delivery really is over a hundred times as costly
as a fast one, that is the right behaviour.

**The question is never "which summary is better".** It is: *what does being wrong actually cost?*

| If being wrong by twice as much costs... | Use | Which gives you |
|---|---|---|
| twice as much | mean absolute error | the **median** |
| four times as much | mean squared error | the **mean** |
| far more than four times | something even more aggressive | a value pulled further into the tail |

## The metric is a choice, and it goes further than two options

Absolute and squared error are two points on a scale. There is a whole family, and one member of it is
worth meeting because it does something neither can.

**Pinball loss** penalises being *under* differently from being *over*:

```
pinball(q) = mean( q * (actual - predicted)      when the prediction is too low
                 (1-q) * (predicted - actual)    when it is too high )
```

At `q = 0.5` the two penalties are equal and it is the absolute error again. At `q = 0.9`, being too low
is nine times worse than being too high.

**Predict before running:** what constant minimises it at `q = 0.9`?

In [ ]:
def pinball(constant, values, q):
    difference = values - constant
    return float(np.mean(np.maximum(q * difference, (q - 1) * difference)))


rows = []
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    losses = np.array([pinball(c, minutes, q) for c in candidates])
    rows.append({"q": q,
                 "best constant": round(float(candidates[losses.argmin()]), 2),
                 "numpy's quantile": round(float(np.quantile(minutes, q)), 2),
                 "reads as": "being %s is %.0fx worse" % ("too low" if q > 0.5 else "too high",
                                                          max(q, 1 - q) / min(q, 1 - q))})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
for q, colour in [(0.1, "#0072B2"), (0.5, "#009E73"), (0.9, "#D55E00")]:
    losses = np.array([pinball(c, minutes, q) for c in candidates])
    ax.plot(candidates, losses, color=colour, linewidth=2.2, label="q = %.1f" % q)
    ax.plot([candidates[losses.argmin()]], [losses.min()], "o", color=colour, markersize=9)
ax.plot(minutes, np.full_like(minutes, 0.4, dtype=float), "|", color="#999999", markersize=13)
ax.set_xlabel("candidate constant (minutes)")
ax.set_ylabel("pinball loss")
ax.set_title("Change what being wrong costs, and the best constant slides along the data",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The best constant slides from 25 minutes to 52 as the penalty for under-promising rises**, and at
`q = 0.5` it lands on the median. The generalisation is exact:

> **The constant that minimises pinball loss at `q` is the `q`-th quantile.** The median is just the case
> `q = 0.5`, where being early and being late cost the same.

Worth checking rather than believing:

In [ ]:
order_statistic = []
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    losses = np.array([pinball(c, minutes, q) for c in candidates])
    position = int(np.ceil(q * len(minutes)))
    order_statistic.append({"q": q, "ceil(q x n)": position,
                            "that many-th smallest": int(np.sort(minutes)[position - 1]),
                            "pinball minimiser": round(float(candidates[losses.argmin()]), 2)})
check = pd.DataFrame(order_statistic)
print(check.to_string(index=False))
print()
print("they agree on all five:", bool((check["that many-th smallest"] == check["pinball minimiser"]).all()))

That is directly useful. A delivery service that promises a time and pays compensation for being late does
**not** want the median - being late costs more than being early, so it wants a higher quantile. Choosing
`q` *is* choosing the trade, and it is a business decision expressed as a number.

**The last two columns disagree, and it is worth knowing why.** At `q = 0.75` the search says 44.00 and
`np.quantile` says 42.50. Neither is wrong; they are two standard definitions of "the q-th quantile".

The pinball minimiser is always **one of the observed values** - specifically the `ceil(q x n)`-th
smallest - because the loss is piecewise linear with kinks only at data points. `np.quantile` instead
*interpolates* between neighbouring values, because it is estimating the quantile of the population the
sample came from rather than minimising a loss on the sample.

## The metric that has no good constant

One more, because it is the most common metric in business reporting and it misbehaves.

**MAPE** - mean absolute *percentage* error - divides each error by the actual value, so a 5-minute miss
on a 22-minute delivery counts more than a 5-minute miss on a 58-minute one.

**Predict before running:** where does its best constant land relative to the mean and the median?

In [ ]:
percentage_error = np.array([np.mean(np.abs(minutes - c) / minutes) for c in candidates])
best_percentage = candidates[percentage_error.argmin()]

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.plot(candidates, percentage_error, color="#7B3294", linewidth=2.2)
for value, colour, label in [(best_percentage, "#7B3294", "MAPE's best: %.1f" % best_percentage),
                             (np.median(minutes), "#009E73", "median: %.0f" % np.median(minutes)),
                             (minutes.mean(), "#D55E00", "mean: %.1f" % minutes.mean())]:
    ax.axvline(value, color=colour, linestyle="--", linewidth=1.8)
    ax.text(value, percentage_error.max() * 0.96, label, rotation=90, ha="right", va="top",
            color=colour, fontsize=9)
ax.set_xlim(20, 55)
ax.set_xlabel("candidate constant (minutes)")
ax.set_ylabel("mean absolute percentage error")
ax.set_title("MAPE prefers a constant below both the median and the mean", fontsize=11.5)
plt.tight_layout()
plt.show()

print("best constant under MAPE   : %.2f" % best_percentage)
print("median                     : %.2f" % np.median(minutes))
print("mean                       : %.2f" % minutes.mean())

**31.0 - below the median and nearly five and a half minutes below the mean.**

The mechanism is division. An error is divided by the *actual* value, so errors on small actuals are
inflated and errors on large actuals are discounted. The cheapest way to keep the inflated errors small is
to **predict low**, and MAPE duly rewards that.

**MAPE systematically prefers under-prediction**, which is a serious problem for a metric widely used to
judge forecasts that feed stock ordering and staffing. It also breaks entirely when an actual is zero, and
it is not symmetric - a prediction twice the actual scores 100%, while a prediction of half the actual
scores 50%.

That is 05-04's subject in full. It is here because it makes the chapter's point sharply: **you cannot
choose a metric on the grounds that it is easy to explain to a stakeholder.** A metric is an instruction
about what to predict, and this one instructs the model to be pessimistic.

## Putting it together

Every metric on this data, and the constant it asks for.

In [ ]:
def best_constant(loss):
    values = np.array([loss(c) for c in candidates])
    return float(candidates[values.argmin()])


summary = pd.DataFrame([
    {"metric": "mean absolute error", "best constant": best_constant(
        lambda c: np.abs(minutes - c).mean()), "which is": "the median"},
    {"metric": "mean squared error", "best constant": best_constant(
        lambda c: ((minutes - c) ** 2).mean()), "which is": "the mean"},
    {"metric": "pinball, q = 0.9", "best constant": best_constant(
        lambda c: pinball(c, minutes, 0.9)), "which is": "the 90th percentile"},
    {"metric": "mean absolute percentage error", "best constant": best_constant(
        lambda c: np.mean(np.abs(minutes - c) / minutes)), "which is": "below both, by construction"},
]).round(2)
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(minutes, np.zeros_like(minutes), "o", color="#cccccc", markersize=10)
for position, (row, colour) in enumerate(zip(summary.itertuples(),
                                             ["#009E73", "#D55E00", "#0072B2", "#7B3294"])):
    ax.plot([row._2], [0], "|", color=colour, markersize=42, linewidth=3)
    ax.annotate("%s\n%.1f" % (row.metric.replace(" error", "").replace("mean ", ""), row._2),
                (row._2, 0), textcoords="offset points",
                xytext=(0, 26 if position % 2 == 0 else -40), ha="center", fontsize=8.5,
                color=colour, fontweight="bold")
ax.set_ylim(-0.9, 0.9)
ax.set_yticks([])
ax.set_xlim(18, 62)
ax.set_xlabel("minutes")
ax.set_title("Four metrics, four different answers, one dataset", fontsize=12)
plt.tight_layout()
plt.show()

**Four metrics, four answers, spread over 21 minutes on a dataset whose middle is 34.** Nothing about the
data changed between them.

That is the chapter, and it is worth stating in the form you will need later:

> **A metric is not a way of scoring a model. It is a specification of what the model should predict.**
> Choose it from the decision the prediction feeds - what does being wrong cost, and is being wrong in one
> direction worse - and the best predictor follows.

## What this means for the rest of module 05

Every model in the next eleven chapters will be compared against a constant. Two consequences:

**The baseline is now precise.** "Beat the baseline" means beat **8.00 minutes of mean absolute error**, or
**99.17 of mean squared error** - whichever metric you committed to. In the units of the target, which is
where a stakeholder can judge it.

**And the comparison must use one metric throughout.** A model tuned on squared error and reported on
absolute error is being judged on a target it was not aiming at - 03-08's loss-versus-metric distinction,
which returns properly in 05-04.

## Common misconceptions

**"The mean is the natural summary, and the median is for skewed data."**
Both are optimal - for different metrics. Which one is "natural" is decided by what being wrong costs.

**"A constant predictor is a formality."**
04-02's per-entity constant beat both a linear regression and a random forest. A constant is a real
predictor and sometimes the winner.

**"Outliers should be removed so the mean behaves."**
Sometimes. But a three-hour delivery is a real delivery, and if it is expensive then a metric that notices
it is the right metric. Removing data to make a summary behave is choosing the answer first.

**"MAPE is fine, it is just a percentage."**
It prefers under-prediction, it explodes near zero, and it is asymmetric. Here it asks for 31 minutes when
the middle of the data is 34.

**"With enough data the mean and the median converge."**
Only if the distribution is symmetric. Delivery times, incomes, claim sizes and durations are not, and the
gap is a property of the distribution, not of the sample size.

**"Choosing a metric comes after building the model."**
The metric determines what the best possible prediction *is*. Choosing it afterwards means you did not
know what you were aiming at.

## Exercises

Solutions: `solutions/05_regression/05-01_baselines_solutions.ipynb`.

### Quick understanding

**E1.** Which constant minimises mean absolute error, and which minimises mean squared error?

**E2.** In one sentence, why does one outlier move the mean much further than the median?

**E3.** What does the constant that minimises pinball loss at `q = 0.9` represent, in words a manager
would understand?

### Hand calculation

**E4.** For the numbers 4, 7, 9, 10, 20: compute the mean and the median. Then compute the mean absolute
error of each as a constant predictor, and confirm which wins.

**E5.** Same numbers. Compute the mean squared error of the mean and of the median, and confirm the other
one wins.

**E6.** For 4, 7, 9, 10, 20, write down the slope of the mean-absolute-error curve just above 9 and just
below 9, using "points above minus points below". Explain what that says about where the minimum is.

**E7.** A bakery loses 1 EUR for every unsold loaf and 4 EUR of goodwill for every customer turned away.
Express that as a pinball `q`, and say which quantile of demand it should bake.

### Coding

**E8.** Write `best_constant(values, loss)` that searches a grid and returns the minimising constant. Use
it to confirm the mean and the median results on the delivery data, then on 200 random draws from a
right-skewed distribution.

**E9.** Reproduce the outlier experiment for a *sequence* of outlier sizes - 60, 100, 200, 500, 1000 -
and plot how far the mean and the median move. What shape is each?

**E10.** Compute the delivery data's best constant under mean **cubed** absolute error,
`mean(|y - c|^3)`. Where does it sit relative to the mean, and what does that suggest about the
progression absolute → squared → cubed?

**E11.** Write `skill(baseline_error, model_error)` and use it to score three candidate constants - 30,
34 and 40 minutes - against the best one, under both MAE and MSE. Which candidate looks worst, and does
the answer depend on the metric?

**E12.** Simulate: draw 1,000 samples of 15 values from a right-skewed distribution, and for each compute
the mean and the median. Plot the two sampling distributions. Which is more variable, and does that
change your advice from E1?

### Interpretation

**E13.** A colleague reports "our average delivery time is 36 minutes" and proposes promising 36 minutes
to customers. Give two reasons that is the wrong promise, using this chapter.

**E14.** A model achieves MAE 7.2 on the delivery data. Compute its skill against the right baseline, and
say whether you would call that a good model.

### Debugging

**E15.** Someone's model reports MAE 8.4 and they conclude it is "close to the baseline". Given this
chapter's numbers, what is wrong with that conclusion?

**E16.** A forecasting system tuned on MAPE consistently under-orders stock. Explain the mechanism and
name a metric that would not do this.

### Exam and interview reasoning

**E17.** "What baseline would you use for a regression problem?" Answer in under a minute, and be ready
for: "why not just always use the mean?"

### Transfer to a different situation

**E18.** You are predicting how long a support ticket will take to resolve, and the team is staffed from
the prediction. Being under-staffed is much worse than being over-staffed. State the metric you would
choose, the constant baseline it implies, and what you would tell the team about the number.

### Explain it to someone non-technical

**E19.** Explain in under 90 words why "what number should we predict?" has more than one right answer,
using the delivery example.

### Optional challenge

**E20.** Show empirically that the constant minimising pinball loss at `q` is the `ceil(q*n)`-th smallest
value, for `n` from 5 to 40 and `q` in {0.1, 0.3, 0.5, 0.7, 0.9}. Report how often the rule holds exactly.
Then explain why the rule involves a ceiling rather than rounding.

In [ ]:
# Your workspace. In memory: minutes, with_breakdown, candidates, absolute_error,
# squared_error, best_absolute, best_squared, pinball, percentage_error, best_constant.

## Mastery check

- [ ] Find the best constant for a metric by grid search, and recognise it as a familiar summary
- [ ] Explain the kink argument for the median and the parabola argument for the mean
- [ ] Predict which summary an outlier moves, and by roughly how much more
- [ ] Convert an asymmetric cost into a pinball `q` and a quantile
- [ ] Say why MAPE prefers under-prediction
- [ ] State a baseline in the units of the target, with its metric named

## What should now feel instinctive

- Asking "judged how?" before asking "predicted how well?"
- Reaching for the median when errors cost proportionally and the mean when they cost quadratically
- Translating "being late is worse than being early" into a quantile rather than a fudge factor
- Quoting a baseline with its metric attached, always
- Distrusting a percentage-based metric on data that can be near zero

## Flashcards

| Front | Back |
|---|---|
| Best constant under absolute error | The median. Here 34.0 minutes, MAE 8.00 |
| Best constant under squared error | The mean. Here 36.4 minutes, MSE 99.17 |
| Why the median | Absolute error is piecewise linear, kinking at each point; the slope is "above minus below" |
| Why the mean | Squared error is a parabola; its minimum is where the average residual is zero |
| One outlier in sixteen | Moved the mean 9.91 minutes and the median 1.0 |
| Pinball loss at q | Minimised by the q-th quantile; the median is q = 0.5 |
| Its minimiser on a sample | The `ceil(q x n)`-th smallest value - always an observed value |
| MAPE's best constant here | 31.0 - below the median, because dividing by the actual rewards predicting low |
| The chapter in one line | A metric is a specification of what to predict, not just a way to score |

## Next

**05-02 · Simple linear regression, fitted by hand.** The constant predictor says the same number for
everybody. The next chapter lets the prediction depend on one feature - a line instead of a level - and
fits it by hand before letting a library do it, so that "least squares" is arithmetic you have done rather
than a function you have called.

The baseline it has to beat is the one this chapter just computed: **8.00 minutes of mean absolute error,
or 99.17 of mean squared error.**